In [18]:
# Why: Add the project root to Python's import path so notebook cells can use project modules.
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

In [19]:
# Why: Import the tools needed to read PDFs, clean and split text, create metadata, and prepare later RAG steps.
# Standard library
import os

# Environment
from dotenv import load_dotenv

# PDF processing
import fitz

# LangChain
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Vector store
from langchain_chroma import Chroma

# LLM
from openai import OpenAI

from ingestion.preprocess import clean_text
from ingestion.chunking import split_into_chunks
from ingestion.metadata import create_metadata

In [20]:
# Why: Confirm which Python environment runs the notebook before installing or importing packages.
import sys

print(sys.executable)

/workspaces/LLM---ZOOMCAMP---RAG---PROJECT--1-/llm-zoomcamp-2026-code/.venv/bin/python


In [21]:
# Why: Verify that pip is available in this notebook's active Python environment.
import subprocess
subprocess.run([sys.executable, "-m", "pip", "--version"])

pip 23.2.1 from /workspaces/LLM---ZOOMCAMP---RAG---PROJECT--1-/llm-zoomcamp-2026-code/.venv/lib/python3.12/site-packages/pip (python 3.12)


CompletedProcess(args=['/workspaces/LLM---ZOOMCAMP---RAG---PROJECT--1-/llm-zoomcamp-2026-code/.venv/bin/python', '-m', 'pip', '--version'], returncode=0)

In [22]:
# Why: Load credentials and configure the API client for any later language-model calls.
load_dotenv()

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [23]:
# Why: Find and open the source PDF, regardless of whether the notebook starts from the project root or its own folder.
pdf_path = "storage/raw/The Universe's Beginning_ A History of Contending Cosmologies from the Steady State to the Hubble Tension.pdf"

pdf_file = os.path.abspath(pdf_path)

# Jupyter may run from the repo root or from the notebook directory
if not os.path.exists(pdf_file):
    pdf_file = os.path.abspath(os.path.join("..", pdf_path))

if not os.path.exists(pdf_file):
    raise FileNotFoundError(f"No such file: {pdf_path}")

doc = fitz.open(pdf_file)

print(len(doc))

21


In [7]:
# Why: Extract the text from every PDF page into one document for preprocessing.
text = ""

for page in doc:
    text += page.get_text()

print(text[:1000])

The Universe's Beginning: A History of
Contending Cosmologies from the Steady
State to the Hubble Tension
The Genesis of Competing Cosmologies: Philosophical
and Historical Context
The mid-twentieth century witnessed a profound shift in humanity's understanding of the
cosmos, marked by a direct conflict between two fundamentally different models of
universal existence: the Steady State theory and the Big Bang theory. This scientific
dispute was not merely a technical disagreement over astronomical data; it was deeply
embedded in the philosophical, methodological, and even theological currents of the era.
To comprehend the full significance of this debate, one must first examine the intellectual
landscape that preceded it, including the epistemic attitudes of key figures, the lingering
influence of classical physics, and the philosophical repugnance many felt toward a
universe with a singular beginning. The very concept of a temporal origin for the cosmos,
or creatio ex nihilo, entered 

In [8]:
# Why: Normalize whitespace so downstream cleaning and chunking operate on consistent text.
text = text.replace("\n", " ")
text = " ".join(text.split())

print(text[:500])

The Universe's Beginning: A History of Contending Cosmologies from the Steady State to the Hubble Tension The Genesis of Competing Cosmologies: Philosophical and Historical Context The mid-twentieth century witnessed a profound shift in humanity's understanding of the cosmos, marked by a direct conflict between two fundamentally different models of universal existence: the Steady State theory and the Big Bang theory. This scientific dispute was not merely a technical disagreement over astronomic


In [9]:
# Why: Remove PDF-specific noise and other unwanted content before creating retrieval chunks.
cleaned_text = clean_text(text)

print(cleaned_text[:500])

The Universe's Beginning: A History of Contending Cosmologies from the Steady State to the Hubble Tension The Genesis of Competing Cosmologies: Philosophical and Historical Context The mid-twentieth century witnessed a profound shift in humanity's understanding of the cosmos, marked by a direct conflict between two fundamentally different models of universal existence: the Steady State theory and the Big Bang theory. This scientific dispute was not merely a technical disagreement over astronomic


In [10]:
# Why: Divide the cleaned document into smaller, searchable pieces suitable for embeddings and retrieval.
chunks = split_into_chunks(cleaned_text)

print(len(chunks))

88


In [24]:
# Why: Inspect one chunk to confirm the splitting step produced useful text segments.
print(chunks[0])

The Universe's Beginning: A History of Contending Cosmologies from the Steady State to the Hubble Tension The Genesis of Competing Cosmologies: Philosophical and Historical Context The mid-twentieth century witnessed a profound shift in humanity's understanding of the cosmos, marked by a direct conflict between two fundamentally different models of universal existence: the Steady State theory and the Big Bang theory. This scientific dispute was not merely a technical disagreement over astronomical data; it was deeply embedded in the philosophical, methodological, and even theological currents of the era


In [17]:
# Why: Add source and chunk details so retrieval results can be traced back to the original PDF.
metadata = create_metadata(chunks, pdf_path)

print(metadata[0])

{'chunk_id': 0, 'document': "The Universe's Beginning_ A History of Contending Cosmologies from the Steady State to the Hubble Tension.pdf", 'source': "storage/raw/The Universe's Beginning_ A History of Contending Cosmologies from the Steady State to the Hubble Tension.pdf", 'text': "The Universe's Beginning: A History of Contending Cosmologies from the Steady State to the Hubble Tension The Genesis of Competing Cosmologies: Philosophical and Historical Context The mid-twentieth century witnessed a profound shift in humanity's understanding of the cosmos, marked by a direct conflict between two fundamentally different models of universal existence: the Steady State theory and the Big Bang theory. This scientific dispute was not merely a technical disagreement over astronomical data; it was deeply embedded in the philosophical, methodological, and even theological currents of the era"}


In [27]:
# Why: Persist the prepared chunks and metadata so the embedding notebook can reuse them.
import json
from pathlib import Path

project_root = Path.cwd().parent
output_file = project_root / "storage" / "processed" / "chunks.json"

output_file.parent.mkdir(parents=True, exist_ok=True)

with output_file.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"Saved to: {output_file}")

Saved to: /workspaces/LLM---ZOOMCAMP---RAG---PROJECT--1-/llm-zoomcamp-2026-code/storage/processed/chunks.json
